In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from evaluate import evaluate
from features import FEATURES, TARGET, load_rated, make_splits

rated = load_rated()
train, valid, test = make_splits(rated)

for name, part in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name:<6} {len(part):>6,} businesses   fail rate {part[TARGET].mean():.2%}")

train  42,834 businesses   fail rate 5.61%
valid  14,278 businesses   fail rate 5.61%
test   14,278 businesses   fail rate 5.61%


In [2]:
rng = np.random.default_rng(0)
results = [evaluate("Random order", valid, rng.random(len(valid)))]
print(pd.DataFrame(results).round(3).to_string(index=False))

       model  recall@20%  PR-AUC (within)  PR-AUC (pooled)  ROC-AUC (pooled)
Random order       0.215            0.067            0.057             0.489


In [3]:
# Learn each business type's fail rate from the TRAINING set only
type_rates = train.groupby("BusinessType")[TARGET].mean()

# Types never seen in training fall back to the overall training fail rate
type_score = valid["BusinessType"].map(type_rates).fillna(train[TARGET].mean())

results.append(evaluate("Business type only", valid, type_score))
print(pd.DataFrame(results).round(3).to_string(index=False))

             model  recall@20%  PR-AUC (within)  PR-AUC (pooled)  ROC-AUC (pooled)
      Random order       0.215            0.067            0.057             0.489
Business type only       0.324            0.078            0.080             0.639


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from features import CATEGORICAL, NUMERIC

SKEWED = ["name_count", "pop_density"]                   # long right tails: log first
OTHER_NUMERIC = [c for c in NUMERIC if c not in SKEWED]

preprocess = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ("log_scaled", make_pipeline(FunctionTransformer(np.log1p), StandardScaler()), SKEWED),
    ("scaled", StandardScaler(), OTHER_NUMERIC),
])

logreg = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(class_weight="balanced", max_iter=2000)),
])
logreg.fit(train[FEATURES], train[TARGET])

lr_score = logreg.predict_proba(valid[FEATURES])[:, 1]
results.append(evaluate("Logistic regression", valid, lr_score))
print(pd.DataFrame(results).round(3).to_string(index=False))

              model  recall@20%  PR-AUC (within)  PR-AUC (pooled)  ROC-AUC (pooled)
       Random order       0.215            0.067            0.057             0.489
 Business type only       0.324            0.078            0.080             0.639
Logistic regression       0.374            0.115            0.151             0.749


In [5]:
import lightgbm as lgb

from features import CATEGORICAL


def lgb_frame(data, reference):
    """Feature table for LightGBM: categories stored as pandas 'category', using the
    category list from the reference (training) data so codes match across splits."""
    X = data[FEATURES].copy()
    for col in CATEGORICAL:
        X[col] = pd.Categorical(X[col], categories=sorted(reference[col].dropna().unique()))
    return X


X_train = lgb_frame(train, train)
X_valid = lgb_frame(valid, train)

lgbm = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
)
lgbm.fit(X_train, train[TARGET])

lgbm_score = lgbm.predict_proba(X_valid)[:, 1]
results.append(evaluate("LightGBM (default settings)", valid, lgbm_score))
print(pd.DataFrame(results).round(3).to_string(index=False))

                      model  recall@20%  PR-AUC (within)  PR-AUC (pooled)  ROC-AUC (pooled)
               Random order       0.215            0.067            0.057             0.489
         Business type only       0.324            0.078            0.080             0.639
        Logistic regression       0.374            0.115            0.151             0.749
LightGBM (default settings)       0.397            0.127            0.152             0.751


In [6]:
from sklearn.model_selection import StratifiedKFold

from features import GROUP
from models import MODELS

# Train + validation combined; the test set stays locked
dev = pd.concat([train, valid]).reset_index(drop=True)
strata = dev[GROUP] + "_" + dev[TARGET].astype(str)

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for fold, (fit_idx, score_idx) in enumerate(folds.split(dev, strata), start=1):
    fit_part, score_part = dev.iloc[fit_idx], dev.iloc[score_idx]
    for name, fit_predict in MODELS.items():
        row = evaluate(name, score_part, fit_predict(fit_part, score_part))
        row["fold"] = fold
        rows.append(row)
    print(f"Fold {fold} done")

cv = pd.DataFrame(rows)

Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done


In [7]:
metrics = ["recall@20%", "PR-AUC (within)", "PR-AUC (pooled)", "ROC-AUC (pooled)"]

summary = cv.groupby("model")[metrics].agg(["mean", "std"]).round(3)
print("Mean and standard deviation across 5 folds:\n")
print(summary.to_string())

# Head to head, fold by fold, on the main metric
per_fold = cv.pivot(index="fold", columns="model", values="recall@20%")
per_fold["LightGBM minus LogReg"] = per_fold["LightGBM (default)"] - per_fold["Logistic regression"]
print("\nrecall@20% per fold:\n")
print(per_fold.round(3).to_string())

Mean and standard deviation across 5 folds:

                    recall@20%        PR-AUC (within)        PR-AUC (pooled)        ROC-AUC (pooled)       
                          mean    std            mean    std            mean    std             mean    std
model                                                                                                      
Business type only       0.316  0.023           0.080  0.002           0.079  0.001            0.635  0.005
LightGBM (default)       0.406  0.016           0.144  0.014           0.153  0.008            0.750  0.007
Logistic regression      0.407  0.029           0.130  0.008           0.156  0.005            0.753  0.006

recall@20% per fold:

model  Business type only  LightGBM (default)  Logistic regression  LightGBM minus LogReg
fold                                                                                     
1                   0.318               0.420                0.438                 -0.018
2             

## Cross-validation results (5 folds on train + validation, test still locked)

| Model | recall@20% | PR-AUC (within) |
|---|---|---|
| Business type only | 0.316 ± 0.023 | 0.080 ± 0.002 |
| Logistic regression | 0.407 ± 0.029 | 0.130 ± 0.008 |
| LightGBM (default) | 0.406 ± 0.016 | 0.144 ± 0.014 |

- **Both models clearly beat the business type baseline** on every fold (+0.06 to +0.13).
  Inspecting the top 20% of each borough's list finds about 41% of failing businesses,
  about double random order (20%).
- **LightGBM vs logistic regression is a tie on recall@20%:** per-fold differences
  -0.018, +0.011, -0.016, +0.019, 0.000. The +0.023 lead on the single validation
  split was luck of the split. This is why models are compared with cross-validation.
- LightGBM shows a small edge on PR-AUC (within), i.e. at the very top of the list,
  but within its own fold-to-fold spread.

### Decision rule (set before tuning)
Tuned LightGBM replaces logistic regression only if it beats it on recall@20%
on at least 4 of 5 folds. Otherwise logistic regression is the final model.